#GenAi Lab 7

**Name : Parth Ladage**


**PRN : 2324001218**

**Roll No : DS63**

### Step 1: Installing Dependencies
We start by installing the official OpenAI Python library, which allows us to interact with the API.

In [ ]:
# Install the OpenAI Python library
!pip install openai --quiet

### Step 2: Initializing the Groq Client
We retrieve our secure API key from Google Colab's secrets manager and configure the OpenAI SDK client to use the Groq API endpoint.

In [ ]:
# Import libraries and create the Groq API client
import os
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

print("Groq client initialized successfully!")

Groq client initialized successfully!


### Step 3: API Connection Test
We perform a simple test request to ensure the connection to the Groq API is established and functioning as expected.

In [ ]:
# Test the Groq API with a simple question
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain supervised learning in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Supervised learning is a machine‑learning approach in which a model is trained on input‑output pairs (labeled data) so it can learn to predict the correct output for new, unseen inputs.


### Step 4: Setting Up System Prompt & Conversation History
We initialize the conversation with a system prompt that guides the behavior and persona of the assistant.

In [ ]:
# Create conversation history with a system message
messages = [
    {
        "role": "system",
        "content": "You are a friendly and concise college lab assistant."
    }
]

print("Conversation initialized.")
print("Number of messages:", len(messages))

Conversation initialized.
Number of messages: 1


### Step 5: Defining the Multi-Turn Conversation Helper
We define a helper function `chat` that appends user messages to the conversation state, calls the model, appends the response, and returns it to preserve state across multiple turns.

In [ ]:
# Define a function that maintains conversation history
def chat(user_input):
    messages.append({
        "role": "user",
        "content": user_input
    })

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages
    )

    reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": reply
    })

    return reply

### Step 6: Testing Context-Aware Conversational Turns
We carry out a continuous three-turn chat session where successive questions rely heavily on the context established in the preceding turns.

In [ ]:
# Ask the first question
print("User: What is supervised learning?")
print("Chatbot:", chat("What is supervised learning?"))
print("Total messages:", len(messages))

User: What is supervised learning?
Chatbot: **Supervised learning** is a type of machine‑learning algorithm that learns a mapping from input data to output labels using a labeled training set.  
- **Input**: feature vectors (e.g., images, sensor readings).  
- **Output**: corresponding labels (e.g., “cat”/“dog”, or a numeric value).  
The algorithm adjusts its parameters to minimize the error between its predictions and the true labels, then generalizes to unseen data.  

Typical examples: image classification, spam detection, and regression tasks like predicting house prices.
Total messages: 3


In [ ]:
# Ask for a real-world example of the previous topic
print("User: Give one real-world example of it.")
print("Chatbot:", chat("Give one real-world example of it."))
print("Total messages:", len(messages))

User: Give one real-world example of it.
Chatbot: **Example: Spam Email Classification**  
A company trains a supervised‑learning model on a dataset of emails that have been manually labeled as “spam” or “not spam.” The model learns patterns (keywords, sender addresses, formatting) that distinguish spam from legitimate mail, then automatically classifies new incoming emails into the two categories.
Total messages: 5


In [ ]:
# Ask which learning method uses labels
print("User: Which of the models we discussed uses labels?")
print("Chatbot:", chat("Which of the models we discussed uses labels?"))
print("Total messages:", len(messages))

User: Which of the models we discussed uses labels?
Chatbot: **All of the models we talked about that are *supervised* use labels.**  
They learn from a training set where each input comes with an associated “ground‑truth” output, e.g.:

| Model | Type of label it needs |
|-------|------------------------|
| Linear / Polynomial regression | Numeric target (regression) |
| Logistic regression, SVM, Naïve Bayes | Class label (classification) |
| Decision tree / Random forest | Class or numeric label |
| Feed‑forward neural network | Class or numeric label |
| Convolutional neural network (for images) | Class label (e.g., “cat”, “dog”) |
| Recurrent neural network (for text) | Class label or sequence label |

In short, **every supervised‑learning model relies on labeled data** to learn the input–output relationship.
Total messages: 7


### Step 7: Checking Complete Conversation State
We print out the cumulative payload of conversation logs representing the full context received by the API.

In [ ]:
# Display the complete conversation history
for message in messages:
    print(message["role"].upper() + ":", message["content"])

SYSTEM: You are a friendly and concise college lab assistant.
USER: What is supervised learning?
ASSISTANT: **Supervised learning** is a type of machine‑learning algorithm that learns a mapping from input data to output labels using a labeled training set.  
- **Input**: feature vectors (e.g., images, sensor readings).  
- **Output**: corresponding labels (e.g., “cat”/“dog”, or a numeric value).  
The algorithm adjusts its parameters to minimize the error between its predictions and the true labels, then generalizes to unseen data.  

Typical examples: image classification, spam detection, and regression tasks like predicting house prices.
USER: Give one real-world example of it.
ASSISTANT: **Example: Spam Email Classification**  
A company trains a supervised‑learning model on a dataset of emails that have been manually labeled as “spam” or “not spam.” The model learns patterns (keywords, sender addresses, formatting) that distinguish spam from legitimate mail, then automatically clas

### Step 8: Simulating Stateless Requests
We iterate over the same questions again, but this time we send them individually without any preceding context to demonstrate how the model reacts when conversational state is lost.

In [ ]:
# Compare answers when previous conversation history is not provided
questions = [
    "What is supervised learning?",
    "Give one real-world example of it.",
    "Which of the models we discussed uses labels?"
]

for question in questions:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            messages[0],
            {"role": "user", "content": question}
        ]
    )

    print("User:", question)
    print("Chatbot:", response.choices[0].message.content)
    print("-" * 50)

User: What is supervised learning?
Chatbot: **Supervised learning** is a type of machine‑learning method where a model is trained on a labeled dataset.  
- **Input (features)** → the data you feed into the model.  
- **Output (label)** → the correct answer you want the model to predict.  

During training the algorithm learns a mapping from inputs to labels by minimizing the error between its predictions and the true labels. After training, it can predict labels for new, unseen inputs.

**Key points:**
- Requires labeled examples (e.g., images tagged “cat” or “dog”).
- Common tasks: classification (assigning categories), regression (predicting a continuous value).
- Examples: spam detection, house‑price prediction, image recognition.

In a lab setting, you’ll usually split data into training, validation, and test sets to evaluate how well the model generalizes.
--------------------------------------------------
User: Give one real-world example of it.
Chatbot: I’d love to help—could yo


### How do answers 2 and 3 depend on answer 1?

Answers 2 and 3 depend on the context established in Answer 1. Since the chatbot maintains conversation history, it can understand follow-up questions without requiring the user to repeat the topic.

- **Answer 2:** Uses the context of Answer 1 to provide a real-world example of supervised learning.
- **Answer 3:** Uses the previous conversation to identify which machine learning models discussed use labeled data.

Without conversation history, the chatbot might not understand what "it" or "the models we discussed" refers to. Therefore, maintaining conversation history makes the chatbot more context-aware and useful.


## Final Observations

1. **The Role of Chat History (State):** Large Language Models (LLMs) are stateless by default. To make them feel "stateful" or conversational, we must append and resend the history of all previous prompts and responses with each new call.
2. **Context Resolution:** In *Step 6*, the model correctly answers "Give one real-world example of **it**" because "it" refers back to "supervised learning".
3. **Failure of Stateless Context:** In *Step 8*, without the chat history, the model cannot resolve what "it" refers to and asks for clarification. Similarly, for the third question, it lists all models generally rather than tailoring the response to "the models we discussed" during our conversation.